# Reporter and analyst Q&A

Writer (Nemotron Ultra) → editor (Ultra) → code checks every number. Q&A (Nemotron Lightning) fetches only the evidence it needs.
Works on any finished supervisor run, read from its audit log in `logs/`. Cells 1-2 cost nothing.

In [ ]:
# 1. Setup
import os, sys, json
from pathlib import Path
from fieldvision_agents.settings import REPO as ROOT
os.chdir(ROOT)

print('working in', Path.cwd())

In [ ]:
# 2. Offline tests (fake LLM, no credits)
from fieldvision_agents.reporter import test_reporter as t
for name, fn in vars(t).items():
    if name.startswith('test_'):
        fn(); print('PASS', name)

In [ ]:
# 3. Pick a finished run: the newest audit log that ended with finish
from fieldvision_agents.reporter.reporter import result_from_log, build_evidence
logs = sorted(Path('logs').glob('audit_*.jsonl'), key=lambda p: p.stat().st_mtime, reverse=True)
runs = [result_from_log(p) for p in logs]
run = next(r for r in runs if r['status'] != 'incomplete')
print(run['run_id'], run['status'], run['policies'], run['critic'])

In [ ]:
# 4. Write the report (Ultra writes, Ultra edits, code checks numbers)
from IPython.display import Markdown, display
from fieldvision_agents.reporter.reporter import write_report
report = write_report(run)
display(Markdown(report['report_md']))
print(json.dumps(report['grounding'], indent=1), '\nsaved to', report['path'])

In [ ]:
# 5. What the editor changed (draft vs final)
import difflib
print('\n'.join(difflib.unified_diff(report['draft_md'].splitlines(), report['report_md'].splitlines(), 'draft', 'final', lineterm='')) or 'No changes')

In [ ]:
# 6. Analyst Q&A (Lightning decides which evidence to fetch)
from fieldvision_agents.reporter.reporter import ask
for q in ['How many silage acres, and how sure are we?',
          'Why did the agent choose that routing policy?',
          'What did the critic flag, and what happened next?',
          'What was the price of corn this year?']:
    a = ask(run, q)
    print(f'Q: {q}\nA: {a["answer"]}\n   sections={a["sections_used"]} grounded={a["grounded"]}\n')